In [4]:
import os
import numpy as np
import pandas as pd

R_EARTH_KM = 6371.0

WORLD_PATH = "data/global_data/worldcities_clean.csv"
CITIES_PATH = "data/compiled_model_ready/MR_cities_with_country_v1.csv"
OUTPUT_PATH = "data/compiled_model_ready/MR_cities_with_country_worldpop_v1.csv"

In [5]:
cities = pd.read_csv(CITIES_PATH)
world = pd.read_csv(WORLD_PATH)

print("Cities:", cities.shape)
print("World:", world.shape)
print(cities.columns)
print(world.columns)

Cities: (509, 15)
World: (47808, 7)
Index(['city', 'country', 'country_norm', 'lat', 'lon', 'crime_index',
       'safety_index', 'crimeindex_2023', 'crimeindex_2020',
       'safetyindex_2020', 'age_0_14', 'age_15_64', 'age_65_plus',
       'population', 'density_per_km2'],
      dtype='object')
Index(['city', 'lat', 'lon', 'country', 'population', 'country_norm',
       'city_norm'],
      dtype='object')


In [6]:
if "country_norm" not in cities.columns:
    cities["country_norm"] = cities["country"].astype(str).str.strip().str.lower()

cities["lat"] = pd.to_numeric(cities["lat"], errors="coerce")
cities["lon"] = pd.to_numeric(cities["lon"], errors="coerce")
cities = cities.dropna(subset=["lat", "lon"]).reset_index(drop=True)
print("Cities after coord clean:", cities.shape)

Cities after coord clean: (509, 15)


In [7]:
# Only rows with valid coords
world = world.dropna(subset=["lat", "lon"]).reset_index(drop=True)

w_lat = world["lat"].values.astype(np.float32)
w_lon = world["lon"].values.astype(np.float32)
w_pop = world["population"].fillna(0).values.astype(np.float32)

print("World non-null:", len(w_lat))

World non-null: 47808


In [8]:
def haversine_vec(lat0, lon0, lats, lons):
    lat0 = np.radians(lat0)
    lon0 = np.radians(lon0)
    lats = np.radians(lats)
    lons = np.radians(lons)

    dlat = lats - lat0
    dlon = lons - lon0

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat0) * np.cos(lats) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return R_EARTH_KM * c

In [9]:
# Pre-filter large cities
LARGE_CITY_POP = 500_000
mask_large = w_pop >= LARGE_CITY_POP
w_lat_large = w_lat[mask_large]
w_lon_large = w_lon[mask_large]
w_pop_large = w_pop[mask_large]
print("Large cities:", len(w_lat_large))

radii = [25.0, 50.0, 100.0]
eps = 1e-3  # to avoid div by zero in gravity term

features = {
    "num_cities_25km": [],
    "sum_pop_25km": [],
    "pop_gravity_25km": [],
    "num_cities_50km": [],
    "sum_pop_50km": [],
    "pop_gravity_50km": [],
    "num_cities_100km": [],
    "sum_pop_100km": [],
    "pop_gravity_100km": [],
    "dist_to_nearest_large_city": [],
    "pop_of_nearest_large_city": [],
}

Large cities: 1437


In [10]:
for idx, row in cities.iterrows():
    lat0 = float(row["lat"])
    lon0 = float(row["lon"])

    # distances to all worldcities
    dists = haversine_vec(lat0, lon0, w_lat, w_lon)

    # For each radius
    for R in radii:
        mask_r = dists <= R
        pop_r = w_pop[mask_r]
        dist_r = dists[mask_r]

        num_key = f"num_cities_{int(R)}km"
        sum_key = f"sum_pop_{int(R)}km"
        grav_key = f"pop_gravity_{int(R)}km"

        features[num_key].append(int(mask_r.sum()))
        features[sum_key].append(float(pop_r.sum()))

        if mask_r.any():
            gravity = float(np.sum(pop_r / ((dist_r + eps) ** 2)))
        else:
            gravity = 0.0
        features[grav_key].append(gravity)

    # Large city proximity
    if len(w_lat_large) > 0:
        dists_large = haversine_vec(lat0, lon0, w_lat_large, w_lon_large)
        min_idx = int(np.argmin(dists_large))
        features["dist_to_nearest_large_city"].append(float(dists_large[min_idx]))
        features["pop_of_nearest_large_city"].append(float(w_pop_large[min_idx]))
    else:
        features["dist_to_nearest_large_city"].append(np.nan)
        features["pop_of_nearest_large_city"].append(np.nan)

    if (idx + 1) % 50 == 0:
        print(f"Processed {idx + 1}/{len(cities)} cities")

Processed 50/509 cities
Processed 100/509 cities
Processed 150/509 cities
Processed 200/509 cities
Processed 250/509 cities
Processed 300/509 cities
Processed 350/509 cities
Processed 400/509 cities
Processed 450/509 cities
Processed 500/509 cities


In [11]:
for k, vals in features.items():
    cities[k] = vals

print("Augmented cities shape:", cities.shape)
print(cities.head())

Augmented cities shape: (509, 26)
           city           country      country_norm        lat         lon  \
0       Caracas         Venezuela         venezuela  10.506093  -66.914601   
1      Pretoria      South Africa      south africa -25.745928   28.187910   
2        Durban      South Africa      south africa -29.861825   31.009909   
3  Port Moresby  Papua New Guinea  papua new guinea  -9.474330  147.159950   
4  Johannesburg      South Africa      south africa -26.205000   28.049722   

   crime_index  safety_index  crimeindex_2023  crimeindex_2020  \
0         83.6          16.4            83.76            84.49   
1         82.0          18.0            76.86            77.49   
2         81.0          19.0            76.86            77.49   
3         80.7          19.3            80.79            81.93   
4         80.7          19.3            76.86            77.49   

   safetyindex_2020  ...  sum_pop_25km  pop_gravity_25km  num_cities_50km  \
0             15.51  ..

In [12]:
cities.to_csv(OUTPUT_PATH, index=False)
print("Saved augmented cities to:", OUTPUT_PATH)

Saved augmented cities to: data/compiled_model_ready/MR_cities_with_country_worldpop_v1.csv


In [1]:
import ipynbname

nb_path = ipynbname.path()
nb_name = nb_path.name 
nb_name

import sys
from pathlib import Path
 
!"{sys.executable}" -m nbconvert --to pdf "{nb_name}"

[NbConvertApp] Converting notebook Geo_Features_v1.ipynb to pdf
[NbConvertApp] Writing 39192 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] Running bibtex 1 time: ['bibtex', 'notebook']
[NbConvertApp] WARNING | b had problems, most likely because there were no citations
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 36088 bytes to Geo_Features_v1.pdf
